<h2>Apply statistical imputation methods such as mean, median, mode, KNN, or iterative imputation to handle missing values across numerical and categorical columns, validating imputation accuracy through data integrity checks and cross-validation.</h2>

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [2]:
medicines = pd.read_csv(r'C:\Users\rutva\Downloads\DS AIML Projects\Daily tasks\Pharmacy management system\archive\medicines.csv')
df_num = medicines[['purchase_price', 'selling_price', 'mrp', 'current_stock']].apply(pd.to_numeric, errors='coerce')

In [3]:
clean_subset = df_num.dropna().copy()
np.random.seed(42)
mask = np.random.rand(*clean_subset.shape) < 0.15
missing_subset = clean_subset.mask(mask)

In [4]:
imputed_mean = missing_subset.fillna(missing_subset.mean())
imputed_median = missing_subset.fillna(missing_subset.median())

In [5]:
knn_imp = KNNImputer(n_neighbors=3)
imputed_knn = pd.DataFrame(knn_imp.fit_transform(missing_subset), columns=clean_subset.columns)

In [6]:
iter_imp = IterativeImputer(random_state=42)
imputed_iter = pd.DataFrame(iter_imp.fit_transform(missing_subset), columns=clean_subset.columns)

In [7]:
results = {}
for name, imp_df in zip(['Mean', 'Median', 'KNN (k=3)', 'Iterative (MICE)'], [imputed_mean, imputed_median, imputed_knn, imputed_iter]):
    rmse = np.sqrt(mean_squared_error(clean_subset[mask], imp_df[mask]))
    mae = mean_absolute_error(clean_subset[mask], imp_df[mask])
    results[name] = {'RMSE': round(rmse, 3), 'MAE': round(mae, 3)}

In [9]:
metrics_df = pd.DataFrame(results).T
print("IMPUTATION ACCURACY BENCHMARK")
print(metrics_df)

IMPUTATION ACCURACY BENCHMARK
                    RMSE    MAE
Mean              19.121  9.375
Median            19.121  9.375
KNN (k=3)         19.121  9.375
Iterative (MICE)   9.132  3.665


In [10]:
cat_imputed = medicines['category'].fillna(medicines['category'].mode()[0])

In [11]:
iter_full = IterativeImputer(random_state=42)
num_imputed = pd.DataFrame(iter_full.fit_transform(df_num), columns=df_num.columns)

In [12]:
medicines_final = num_imputed.copy()
medicines_final['category'] = cat_imputed.values

In [15]:
print("DATA INTEGRITY CHECK")
print("Missing values remaining:", medicines_final.isnull().sum().sum())
print("Data shape preserved:", medicines_final.shape)

DATA INTEGRITY CHECK
Missing values remaining: 0
Data shape preserved: (30, 5)


In [16]:
medicines_final.to_csv('medicines_imputed_validated.csv', index=False)
print("Clean dataset exported successfully as 'medicines_imputed_validated.csv'!")

Clean dataset exported successfully as 'medicines_imputed_validated.csv'!
